In [ ]:
import numpy as np

# top-p sampling
def top_p_sampling(probs, p=0.9):
    """
    probs: 1D numpy array of probabilities (must sum to 1)
    p: cumulative probability threshold
    """
    # Sort indices by probability (descending)
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]

    # Compute cumulative sum
    cumulative_probs = np.cumsum(sorted_probs)

    # Find cutoff
    cutoff = cumulative_probs <= p

    # Ensure at least one token is included
    cutoff[np.argmax(cumulative_probs > p)] = True

    # Keep only top-p tokens
    filtered_indices = sorted_indices[cutoff]
    filtered_probs = probs[filtered_indices]

    # Renormalize
    filtered_probs = filtered_probs / filtered_probs.sum()

    # Sample
    chosen_index = np.random.choice(filtered_indices, p=filtered_probs)

    return chosen_index

# common tweaks: combine with temperature
# probs = probs ** (1 / temperature)
# probs = probs / probs.sum()

In [ ]:
# single head attention (numpy)
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)  # for numerical stability
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def single_head_attention(Q, K, V, mask=None):
    """
    Q: (batch, seq_len_q, d_k)
    K: (batch, seq_len_k, d_k)
    V: (batch, seq_len_k, d_v)
    mask: (batch, seq_len_q, seq_len_k) or None
    """
    d_k = Q.shape[-1]

    # 1. Compute attention scores
    scores = np.matmul(Q, K.transpose(0, 2, 1))  # (batch, seq_q, seq_k)
    scores = scores / np.sqrt(d_k)

    # 2. Apply mask (optional)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # 3. Softmax to get attention weights
    attn_weights = softmax(scores, axis=-1)

    # 4. Weighted sum of values
    output = np.matmul(attn_weights, V)  # (batch, seq_q, d_v)

    return output, attn_weights